# Week 3: Data Preprocessing & Cleaning
**Author:** Sakshitha  
**Goal:** Clean, scale, split data, and apply SMOTE to balance classes.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
project_path = '/content/drive/MyDrive/ParkinsonsProject/ParkinsonsDetection'
os.chdir(project_path)

import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

Mounted at /content/drive


## Load Raw Data and Split Indices

In [3]:
df = pd.read_csv('data/raw/parkinsons.csv')
train_idx = np.load('data/processed/train_indices.npy')
test_idx = np.load('data/processed/test_indices.npy')

## Separate Features and Target
Drop the `name` column (identifier). The `subject_id` is not needed for modeling.

In [4]:
X = df.drop(['status', 'name'], axis=1)
y = df['status']

## Handle Missing Values (if any)
This dataset has no missing values, but we include a safe check.

In [5]:
if X.isnull().sum().any():
    from sklearn.impute import SimpleImputer
    imputer = SimpleImputer(strategy='median')
    X.iloc[train_idx] = imputer.fit_transform(X.iloc[train_idx])
    X.iloc[test_idx] = imputer.transform(X.iloc[test_idx])
    print("Missing values imputed with median.")
else:
    print("No missing values found.")

No missing values found.


## Feature Scaling
Fit scaler on training data only, then transform both train and test.

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X.iloc[train_idx])
X_test_scaled = scaler.transform(X.iloc[test_idx])

# Save scaler for later use in GUI
joblib.dump(scaler, 'models/scaler.joblib')
print("Scaler saved to models/scaler.joblib")

Scaler saved to models/scaler.joblib


## Create Train/Test Sets

In [7]:
X_train = X_train_scaled
X_test = X_test_scaled
y_train = y.iloc[train_idx].values
y_test = y.iloc[test_idx].values

## Handle Class Imbalance with SMOTE
Apply SMOTE **only on the training set** to avoid data leakage.

In [8]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Original class counts:", np.bincount(y_train))
print("After SMOTE:", np.bincount(y_train_res))

Original class counts: [36 98]
After SMOTE: [98 98]


## Save Processed Data
Save arrays as `.npy` files for use in later weeks.

In [10]:
np.save('data/processed/X_train.npy', X_train_res)
np.save('data/processed/X_test.npy', X_test)
np.save('data/processed/y_train.npy', y_train_res)
np.save('data/processed/y_test.npy', y_test)

print(" Processed data saved to data/processed/")

 Processed data saved to data/processed/


## Done
Now we have balanced, scaled training data and untouched test data.  
Proceed to Week 4 (Feature Selection).